# Division Result Viewer

Load `results/division_result_16.json` and render spherical triangles on the octant.


In [ ]:
import numpy as np
from pathlib import Path

from coordinate_fileio import load_division_result, validate_division_result
from sphere_division_algorithms import build_octant_triangle_keys, build_projected_positions
from sphere_division_visualization import plot_octant_mesh_from_positions_with_area_color

In [ ]:
result_path = Path('results') / 'division_result_16.json'
report = validate_division_result(result_path)
is_valid = bool(report['valid'])

print(f'loaded: {result_path.as_posix()}')
print(f"validation: {'OK' if is_valid else 'NG'}")

if not is_valid:
    print('Invalid division result file. Please fix the following:')
    if not report['counts']['ok']:
        c = report['counts']
        print(f"- count mismatch: stored={c['stored_points']} expected={c['expected_stored_points']}, expanded={c['expanded_points']} expected={c['expected_expanded_points']}")
    if not report['index_check']['ok']:
        print(f"- invalid index records: {report['index_check']['error_count']}")
    if not report['sphere_constraint']['ok']:
        print(f"- sphere constraint violations: {report['sphere_constraint']['violation_count']}")
    if not report['arc_constraint']['ok']:
        print(f"- arc constraint violations: {report['arc_constraint']['violation_count']}")

N, positions = load_division_result(result_path)
triangle_keys = build_octant_triangle_keys(N)
point_count = int(np.count_nonzero(~np.isnan(positions[:, :, 0])))
print(f'N={N}, point_count={point_count}, triangle_count={len(triangle_keys)}')

In [ ]:
# Before area optimization (initial octant lattice projected to constraints)
if not is_valid:
    print('Skip before-plot because validation failed.')
else:
    positions_before = build_projected_positions(N)

    fig_before, ax_before, areas_before = plot_octant_mesh_from_positions_with_area_color(
        triangle_keys=triangle_keys,
        positions=positions_before,
        n=N,
        colormap='bwr',
        save_path=Path('figures') / f'octant_mesh_before_area_optimizer_{N}.svg',
    )
    print('before optimization area stats:')
    print(f'  min={areas_before.min():.10f}, max={areas_before.max():.10f}, std={areas_before.std(ddof=0):.10f}')

In [ ]:
# After area optimization (loaded from result file)
if not is_valid:
    print('Skip after-plot because validation failed.')
else:
    fig_after, ax_after, areas_after = plot_octant_mesh_from_positions_with_area_color(
        triangle_keys=triangle_keys,
        positions=positions,
        n=N,
        colormap='bwr',
        save_path=Path('figures') / f'octant_mesh_after_area_optimizer_{N}.svg',
    )
    print('after optimization area stats:')
    print(f'  min={areas_after.min():.10f}, max={areas_after.max():.10f}, std={areas_after.std(ddof=0):.10f}')
